# Zero-shot extraction: Jantzen species interactions

Use each converted Jantzen paper's MinerU Markdown as the source. The model returns records defined by `src/standard/jantzen_species_interaction.py`. A progress bar tracks completed papers, while results are saved after each paper so rerunning skips completed papers. Evaluation is outside this notebook.

Model: SURF `Qwen/Qwen3.6-27B-FP8` (`qwen27b` alias). Set `SURF_API_KEY` in `.env` before running.

This run adds explanations for all 71 interaction labels. Restart the kernel before running so it loads the new prompt. Results from the earlier rules_v2 run remain in their original output folder.


In [ ]:
import csv
import sys
from pathlib import Path

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.output import ExtractionOutput
from src.model import StructuredLLMClient, load_provider_config
from src.zeroshot import PROMPT_FINGERPRINT, find_jantzen_markdown, run_extraction


In [ ]:
MARKDOWN_DIR = PROJECT_ROOT / 'data/ready/jantzen_species_interaction'
OUTPUT_DIR = PROJECT_ROOT / 'outputs/zeroshot/jantzen_qwen36_27b_fp8_rules_v3'

PROVIDER = 'surf'
MODEL = 'qwen27b'  # SURF model ID: Qwen/Qwen3.6-27B-FP8
LIMIT = None  # Set to 1 for a one-paper trial; None selects all Markdown papers.
RESPONSE_MODE = 'json_schema'
MAX_OUTPUT_TOKENS = 8_000
OVERWRITE = False


In [ ]:
papers = find_jantzen_markdown(MARKDOWN_DIR)
if LIMIT is not None:
    papers = papers[:LIMIT]
print(f'Selected {len(papers)} Markdown papers')
papers[:5]


In [ ]:
config = load_provider_config(PROVIDER, MODEL)
output = ExtractionOutput(OUTPUT_DIR)
output.write_config({
    'provider': config.provider,
    'model': config.model,
    'base_url': config.base_url,
    'response_mode': RESPONSE_MODE,
    'max_output_tokens': MAX_OUTPUT_TOKENS,
    'source_dir': str(MARKDOWN_DIR),
    'prompt_fingerprint': PROMPT_FINGERPRINT,
})
summary = run_extraction(
    papers,
    client=StructuredLLMClient(config),
    output=output,
    overwrite=OVERWRITE,
    response_mode=RESPONSE_MODE,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)
summary


In [ ]:
with output.status_path.open(encoding='utf-8', newline='') as handle:
    status = list(csv.DictReader(handle))
with summary.records_path.open(encoding='utf-8', newline='') as handle:
    records = list(csv.DictReader(handle))
print(f'{len(records)} interaction records; {summary.failed} papers failed')
print(f'Records: {summary.records_path}')
print(f'Per-paper JSON: {output.raw_dir}')
status[:5], records[:5]
